# SENTINEL — YOLOv8s Weapon FINE-TUNE v2.2 on Kaggle (Phase 3, dataset v2.1)

Dataset v2.1 par **v2 weights se fine-tune** (scratch training nahi) — **v2.2 recipe**: 40 epochs, learning rate v2.1 se aadha (lr0 0.0005), warmup 5 epochs, patience 12, rotation + perspective + blur/weather augmentation, mosaic aakhri 10 epochs band. **2x T4** GPUs par andazan ~2 ghante, aik session mein.

**v2.1 se sabak (run weapon_yolov8s_v21_20261120_1836):** lr0 0.001 par val mAP50 pehle 3 epochs mein 0.74 se 0.61 tak crater ho gaya, phir 25 epochs mein sirf 0.82 tak recover hua — frozen test 0.8108 vs v2 baseline 0.8621 (gate FAIL). Hard frames par v2.1 zabardast tha (FPs −89%), is liye data wahi hai — sirf recipe halki aur lambi hai: kam LR + lamba warmup (crater rokne ke liye) + zyada epochs (recover karne ke liye).

Dataset mix (script `scripts/data_preparation/assemble_v21_dataset.py` se bana): v2.1 hard examples (mined frames + 1,177 Roboflow knife images + weather/blur variants) + **50% v2 train replay** (catastrophic forgetting se bachne ke liye). Val 1,351 aur **test 680 FROZEN** (v2 wali hi split) — naye weights sirf tab adopt honge jab frozen test par v2 baseline (mAP50 0.8621) ko hara dein, warna v2 weights hi fallback rahenge.

**Pehle ye 4 cheezein set karo (notebook settings, right sidebar):**
1. **Accelerator: GPU T4 x2** select karo (P100 nahi — wo 1 GPU hai).
2. **Internet: ON** rakho (pip install + repo clone + Drive weights download ke liye).
3. **Add Input → dataset `sentinel-weapon-v21`** attach karo (private dataset — review ho chuka hai, Kaggle par maujood hai).
4. **v2 starting weights Drive se aati hain** — Step 3 unhe `SENTINEL_FYP/runs/weapon_yolov8s_v2_20261119_1059/` se download karke SHA-256 se verify karta hai (koi manual upload nahi chahiye).

**Phir: Save & Run All** dabao. Kisi cell mai red error aye to **wahin ruk jao** aur output Muse ko bhejo, khud fix mat karo. Step 5 dataset ke `manifest.json` se review status check karta hai — labels UNREVIEWED hon to training wahin ruk jayegi.

Training script repo se ata hai (`scripts/training/train_weapon_detector.py`) — notebook use clone karke chalati hai, taake Kaggle aur repo hamesha ek hi code use karein.

In [ ]:
# ============ CONFIG — sirf yahi cell badalna hai ============
RUN_MODE = "v22"        # "v22" = 40-epoch fine-tune (v2.1 se halki recipe) | "smoke" = 3 epochs (pipeline check)
REPO_BRANCH = "main"   # training script main se clone hota hai
OVERSAMPLE_KNIFE = 1  # v2.1 mein oversampling BAND (1) — mix mein pehle se 1,177 Roboflow knife images hain, dobara 2x karne se Knife over-weight ho jayega

from datetime import datetime
from pathlib import Path

EPOCHS = 3 if RUN_MODE == "smoke" else 40
RUN_NAME = f"weapon_yolov8s_{RUN_MODE}_{datetime.now():%Y%m%d_%H%M}"

# ---- Fine-tune hyperparameters (v2 run: lr0 0.01, degrees 0, perspective 0, 100 epochs) ----
LR0 = 0.0005         # v2.1 se aadha — v2.1 mein lr0 0.001 par val mAP50 epochs 2-3 mein 0.74->0.61 crater hua tha
DEGREES = 20         # rotation +/-20 deg (research: 15-30; vertical-knife->gun failure ka fix)
PERSPECTIVE = 0.0005 # tilted camera (research: 0.0005-0.001)
CLOSE_MOSAIC = 10    # aakhri 10 epochs mosaic band (small objects ke liye)
MIXUP = 0.1          # research: cheap test (v2 mein 0.0 tha)
ERASING = 0.4        # occlusion (v2 default bhi 0.4)
WARMUP = 5.0         # v2.1 mein default 3.0 tha — lamba warmup pretrained features ko shuru ke shock se bachata hai
PATIENCE = 12        # 40-epoch run ke liye (v2.1 mein 10 tha, 25 epochs par)

# ---- v2 starting weights (Drive: SENTINEL_FYP/runs/weapon_yolov8s_v2_20261119_1059/) ----
V2_WEIGHTS_DRIVE_ID = "1z2baegmmqL1frskwzWQu5c6ffwDwj604"
V2_WEIGHTS_SHA256 = "1c6ef07711287d62a8d010ffe9d9d7a7b558f632f285e91e1b4f66cd9645c847"
MODEL_PATH = "/kaggle/working/weapon_yolov8s_v2_best.pt"

# ---- v2 baseline (frozen test) — adoption gate: naya model sirf tab adopt hoga jab mAP50 is se zyada ho ----
V2_BASELINE_MAP50 = 0.8621
V2_BASELINE_GUN_AP50 = 0.7813
V2_BASELINE_KNIFE_AP50 = 0.943

DATASET_SLUG = "sentinel-weapon-v21"  # Add Input se attach hona chahiye

# Dataset ka path hardcode nahi hai — Step 2 usay /kaggle/input ke andar
# structure (images/ + labels/) se dhoondta hai, kyunki Kaggle ka mount path
# mukhtalif ho sakta hai (/kaggle/input/<slug>/ ya /kaggle/input/datasets/...).
# /kaggle/input read-only hai, is liye Step 5 dataset ko WORK mein copy karta hai.
WORK = Path("/kaggle/working/datasets")
RUNS_PROJECT = "/kaggle/working/runs/weapon"
OUTPUT_DIR = Path("/kaggle/working/output") / RUN_NAME   # best.pt + results yahan — run ke baad download kar lena

print("RUN_MODE :", RUN_MODE, "| EPOCHS:", EPOCHS)
print("RUN_NAME :", RUN_NAME)
print(f"Fine-tune: lr0={LR0} warmup={WARMUP} degrees={DEGREES} perspective={PERSPECTIVE} close_mosaic={CLOSE_MOSAIC} mixup={MIXUP} erasing={ERASING} patience={PATIENCE}")
print("Starting weights: v2 best.pt (Drive) — SHA-256 verify Step 3 mein hoga")

## Step 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU nahi mila — notebook settings mai Accelerator = GPU T4 x2 select karo, phir dobara chalao."
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, "|", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
# 2 GPUs hon to DDP (device 0,1, batch 32); 1 GPU ho to single-GPU (device 0, batch 16, ~13 ghante)
DEVICE = "0,1" if N_GPU >= 2 else "0"
BATCH = 32 if N_GPU >= 2 else 16
print("DEVICE:", DEVICE, "| BATCH:", BATCH)

## Step 2 — Dataset dhoondna (Add Input se `sentinel-weapon-v21` attach hona chahiye)

In [ ]:
# Kaggle datasets ko mukhtalif jagah mount kar sakta hai (/kaggle/input/<slug>/
# ya /kaggle/input/datasets/...), is liye path hardcode karne ki bajaye
# structure se dhoondo: wo folder jiske andar images/ aur labels/ dono hon.
INPUT_ROOT = Path("/kaggle/input")
cands_src = [
    p.parent
    for p in INPUT_ROOT.rglob("images")
    if p.is_dir() and (p.parent / "labels").is_dir()
]
assert cands_src, f"Dataset nahi mila: /kaggle/input ke andar images/+labels/ wala folder nahi — right sidebar mai Add Input se dataset '{DATASET_SLUG}' attach karo, phir dobara chalao."
DATASET_SRC = sorted(cands_src, key=lambda p: len(p.parts))[0]
print("Dataset mila:", DATASET_SRC)

## Step 3 — Ultralytics install + v2 starting weights Drive se (SHA-256 verify ke saath)

In [ ]:
%pip install -q ultralytics gdown
import hashlib

# v2 best.pt Drive se download karo (file id run log se — Drive path:
# SENTINEL_FYP/runs/weapon_yolov8s_v2_20261119_1059/weapon_yolov8s_v2_20261119_1059_best.pt).
# gdown apni progress khud dikhata hai; agar ye fail ho to Drive file ki
# link-sharing check karo, ya best.pt manually /kaggle/working mein upload
# karke MODEL_PATH wahi rakh do aur cell dobara chalao (verify phir bhi hoga).
print("Downloading v2 starting weights from Drive ...", flush=True)
!gdown --id $V2_WEIGHTS_DRIVE_ID -O $MODEL_PATH

size = Path(MODEL_PATH).stat().st_size
print("v2 best.pt size (MB):", round(size / 1e6, 2))
assert size > 20_000_000, "Weights file adhoori lagti hai — ye cell dobara chalao."
sha = hashlib.sha256(Path(MODEL_PATH).read_bytes()).hexdigest()
print("SHA-256:", sha)
assert sha == V2_WEIGHTS_SHA256, "SHA-256 MISMATCH — galat ya corrupt weights file! Training roki gayi. Ye output Muse ko bhejo."
print("v2 starting weights VERIFIED (SHA-256 match)")

## Step 4 — Repo clone (training script wahi se ayega)

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/sentinel_repo
!git clone -b $REPO_BRANCH --single-branch https://github.com/muddassir1935/FYP.git /kaggle/working/sentinel_repo

script = Path("/kaggle/working/sentinel_repo/scripts/training/train_weapon_detector.py")
assert script.exists(), "Training script nahi mila — branch ka naam CONFIG mai check karo, Muse ko batao."
print("Training script ready:", script)

## Step 5 — Dataset copy + verification (manifest se review status + counts check — UNREVIEWED ho ya counts galat hon to training yahi ruk jayegi)

In [ ]:
import json
import shutil

WORK.mkdir(parents=True, exist_ok=True)
# /kaggle/input read-only hai aur oversampling (Step 5b) ko likhna hota hai,
# is liye dataset ko WORK mein copy karo. Pehle se copy ho to fresh copy banao
# (purani copy mein oversample ki hui _os1 files bhi ho sakti hain).
dest = WORK / "weapon_yolo_v21"
if dest.exists():
    print("Purani copy mili, fresh copy bana raha hun...")
    shutil.rmtree(dest)
# Poora dataset copy karne ki bajaye symlinks banao: Kaggle ka input mount
# chhoti files ka data copy karte hue bahut slow hai (v7 run isi copy par ruk
# gaya), jab ke training ko sirf parhna hota hai — symlinks se kaam chal jata
# hai. Oversampling phir bhi ASLI _os1 files likhta hai.
# NOTE: copytree(symlinks=True) sirf pehle se mojood symlinks ko bachata hai,
# naye symlinks NAHI banata — is liye copy_function=os.symlink zaroori hai.
import os
linked = [0]
def _link(src, dst):
    os.symlink(src, dst)
    linked[0] += 1
    if linked[0] % 5000 == 0:
        print(f"  ... {linked[0]} files linked", flush=True)
print(f"Linking (symlinks) {DATASET_SRC} -> {dest} ...")
shutil.copytree(DATASET_SRC, dest, copy_function=_link)
print(f"Linking ho gaya: {linked[0]} files.")

# dataset root = wo folder jiske andar images/ aur labels/ dono hon (zip ki structure dono surton mai handle hai)
cands = [p.parent for p in WORK.rglob("images") if (p.parent / "labels").exists()]
assert cands, "Dataset structure samajh nahi aya — ye output Muse ko bhejo."
root = sorted(cands, key=lambda p: len(p.parts))[0]
print("Dataset root:", root)

# ---- manifest check: review status + expected counts + pHash audit ----
manifest_path = root / "manifest.json"
assert manifest_path.exists(), "manifest.json nahi mila — ye dataset assemble_v21_dataset.py se nahi bana. Ye output Muse ko bhejo."
manifest = json.loads(manifest_path.read_text())
review_status = manifest.get("review_status")
print("Manifest review_status:", review_status)
assert review_status == "reviewed", "LABELS ABHI UNREVIEWED HAIN — user ka Drive review mukammal hone aur dataset dobara assemble hone se pehle training NAHI ho sakti. Ruk jao, Muse ko batao."
audit = manifest.get("audit", {})
print("pHash audit verdict:", audit.get("verdict"), f"(leakage: {audit.get('leakage_count')})")
assert audit.get("verdict") == "CLEAN", "pHash audit CLEAN nahi hai — frozen test mein leakage hai. Training roki gayi. Ye output Muse ko bhejo."
final = manifest.get("final", {})
print("Train breakdown:", json.dumps(final.get("train_breakdown", {})))

IMG_EXTS = {".jpg", ".jpeg", ".png"}
expected = {"train": final["train"], "val": final["val"], "test": final["test"]}
# val 1,351 aur test 680 v2 se FROZEN hain — in mein koi tabdeeli nahi honi chahiye
assert expected["val"] == 1351 and expected["test"] == 680, f"Val/test FROZEN counts galat hain (val {expected['val']}, test {expected['test']}) — ye output Muse ko bhejo."
ok = True
for split, exp in expected.items():
    imgs = [p for p in (root / "images" / split).glob("*") if p.suffix.lower() in IMG_EXTS]
    lbls = list((root / "labels" / split).glob("*.txt"))
    print(f"{split}: images = {len(imgs)} (expected {exp}), labels = {len(lbls)}")
    if len(imgs) != exp:
        ok = False

# labels sanity: har line ki pehli value (class id) sirf 0 (Gun) ya 1 (Knife) ho
# Poora check local machine par pehle se ho chuka hai (assembly script har
# label validate karta hai) aur Kaggle wali files wahi hain (same zip se
# extract hui), is liye yahan SAMPLE check kaafi hai — Kaggle ke slow mount
# par saari files parhna bahut slow hai (v9 run isi par atak gaya). Har 12th
# file check karo (~2,000 files, teeno splits se). Koi systematic kharabi
# hoti to sample mein pakri jati.
all_lbl = sorted((root / "labels").rglob("*.txt"))
sample = all_lbl[::12]
bad, checked = 0, 0
for i, txt in enumerate(sample, 1):
    if i % 500 == 0:
        print(f"  ... {i}/{len(sample)} sample label files checked ({checked} lines)", flush=True)
    for line in txt.read_text().splitlines():
        if line.strip():
            checked += 1
            if int(float(line.split()[0])) not in (0, 1):
                bad += 1
print(f"label lines checked = {checked} (sample {len(sample)}/{len(all_lbl)} files), bad class ids = {bad}")

assert ok and bad == 0, "DATASET VERIFICATION FAILED — training roki gayi. Ye poora output Muse ko bhejo."

# Kaggle ke liye data yaml (repo wali yaml local paths ke liye hai, ye wali /kaggle/working ke absolute path ke saath)
DATA_YAML = root / "sentinel_data.yaml"
DATA_YAML.write_text(f"path: {root}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 2\nnames: ['Gun', 'Knife']\n")
print("DATA_YAML:", DATA_YAML)
print("DATASET VERIFIED OK")

## Step 5b — Knife oversampling (v2.1 mein BAND hai)

v2 run mein Knife 2x oversample hua tha (Gun:Knife ~9:1 → ~4.5:1). v2.1 mix mein pehle se 1,177 Roboflow knife images + v2.1 knife boxes hain, is liye dobara oversampling se Knife over-weight ho jayega — `OVERSAMPLE_KNIFE = 1` rakha hai aur ye cell sirf ye print karke guzar jayega. Agar manifest ke counts dekh kar phir bhi oversampling chahiye ho to CONFIG mein `OVERSAMPLE_KNIFE = 2` karke dobara chalao.

In [ ]:
# Step 5b — Knife oversampling: knife wali har image ki (OVERSAMPLE_KNIFE - 1) copies
import shutil

assert OVERSAMPLE_KNIFE >= 1, "OVERSAMPLE_KNIFE kam az kam 1 hona chahiye (1 = oversampling band)"
if OVERSAMPLE_KNIFE == 1:
    print("Oversampling band hai (OVERSAMPLE_KNIFE = 1).")
else:
    train_img = root / "images" / "train"
    train_lbl = root / "labels" / "train"
    knife_imgs, made, skipped, scanned = 0, 0, 0, 0
    for lbl in sorted(train_lbl.glob("*.txt")):
        scanned += 1
        if scanned % 5000 == 0:
            print(f"  ... {scanned} train labels scanned, {knife_imgs} knife images mili", flush=True)
        if any(lbl.stem.endswith(s) for s in ("_os1", "_os2", "_os3")):
            continue  # pehle se bani copy — dobara duplicate mat karo
        has_knife = any(
            line.strip() and int(float(line.split()[0])) == 1
            for line in lbl.read_text().splitlines()
        )
        if not has_knife:
            continue
        knife_imgs += 1
        img = None
        for ext in (".jpg", ".jpeg", ".png"):
            cand = train_img / (lbl.stem + ext)
            if cand.exists():
                img = cand
                break
        if img is None:
            skipped += 1
            continue  # image nahi mili to label akela copy karne ka faida nahi
        for k in range(1, OVERSAMPLE_KNIFE):
            dst_lbl = train_lbl / f"{lbl.stem}_os{k}.txt"
            if dst_lbl.exists():
                continue  # cell dobara chale to copies dobara na banein
            shutil.copy2(lbl, dst_lbl)
            shutil.copy2(img, train_img / f"{lbl.stem}_os{k}{img.suffix}")
            made += 1
            if made % 500 == 0:
                print(f"  ... {made} copies ban gayi", flush=True)
    total = len([p for p in train_img.glob("*") if p.suffix.lower() in IMG_EXTS])
    print(f"knife images found = {knife_imgs}, copies made = {made}, skipped (no image) = {skipped}")
    print(f"train images ab = {total} (oversampling se pehle {final['train']})")
    print("OVERSAMPLING DONE")


## Step 6 — Fine-tune training (v2.2, 40 epochs, 2x T4 par andazan ~2 ghante — Save & Run All se chalao, session 12h cap hai)

In [ ]:
%cd /kaggle/working/sentinel_repo
print("Fine-tune start: v2 weights -> v2.1 mix (v2.2 recipe) | "
      f"epochs={EPOCHS} lr0={LR0} warmup={WARMUP} degrees={DEGREES} perspective={PERSPECTIVE} "
      f"close_mosaic={CLOSE_MOSAIC} mixup={MIXUP} erasing={ERASING} patience={PATIENCE} "
      f"| batch={BATCH} device={DEVICE} (Ultralytics har epoch ki progress khud print karega)", flush=True)
!python scripts/training/train_weapon_detector.py --data $DATA_YAML --model $MODEL_PATH --epochs $EPOCHS --imgsz 640 --batch $BATCH --device $DEVICE --workers 4 --patience $PATIENCE --lr0 $LR0 --warmup-epochs $WARMUP --degrees $DEGREES --perspective $PERSPECTIVE --close-mosaic $CLOSE_MOSAIC --mixup $MIXUP --erasing $ERASING --project $RUNS_PROJECT --name $RUN_NAME
print("Training cell khatam — agla cell (Step 7) test metrics dikhayega.", flush=True)

## Step 7 — Results (frozen test per final numbers + v2 baseline se comparison + adoption gate)

In [ ]:
import json
from IPython.display import Image, display

run_dir = Path(RUNS_PROJECT) / RUN_NAME
summary = json.loads((run_dir / "run_summary.json").read_text())
print("TEST METRICS:", json.dumps(summary["test_metrics"], indent=2))

# v2 baseline se comparison — adoption gate (research risk table):
# naya checkpoint sirf tab adopt hota hai jab frozen test mAP50 v2 (0.8621) se zyada ho.
tm = summary["test_metrics"]
new_map50 = tm.get("mAP50")
print(f"\nBaseline v2 (frozen test): mAP50 = {V2_BASELINE_MAP50} | Gun AP50 = {V2_BASELINE_GUN_AP50} | Knife AP50 = {V2_BASELINE_KNIFE_AP50}")
print(f"Fine-tune v2.1 (frozen test): mAP50 = {new_map50} | " + " | ".join(f"{k} AP50 = {v.get('AP50')}" for k, v in tm.get("per_class", {}).items()))
if new_map50 is not None and new_map50 > V2_BASELINE_MAP50:
    print(f"GATE: PASS — v2.1 beats v2 by {new_map50 - V2_BASELINE_MAP50:.4f} mAP50. Naye weights adopt karne ke candidate hain (phir bhi pehle webcam/bullet-video retest hoga).")
elif new_map50 is not None:
    print(f"GATE: FAIL — v2.1 ({new_map50}) v2 baseline ({V2_BASELINE_MAP50}) se neeche hai. v2 weights hi fallback rehte hain; run log mein darj karo aur Muse ko batao.")
else:
    print("GATE: UNKNOWN — test metrics nahi mile. Ye output Muse ko bhejo.")

cm = run_dir / "confusion_matrix.png"
if cm.exists():
    display(Image(str(cm), width=640))

## Step 8 — Weights + results output folder mein save (run ke baad download kar lena + COPY block Muse ko bhejna hai)

In [ ]:
import hashlib
import shutil

dest = OUTPUT_DIR
dest.mkdir(parents=True, exist_ok=True)

best = run_dir / "weights" / "best.pt"
dated = dest / f"{RUN_NAME}_best.pt"
shutil.copy(best, dated)
for extra in ["results.csv", "confusion_matrix.png", "results.png", "run_summary.json"]:
    srcp = run_dir / extra
    if srcp.exists():
        shutil.copy(srcp, dest / extra)
# last.pt bhi rakh lo — agar kabhi resume karna pare to kaam ayega
last = run_dir / "weights" / "last.pt"
if last.exists():
    shutil.copy(last, dest / f"{RUN_NAME}_last.pt")

sha = hashlib.sha256(dated.read_bytes()).hexdigest()
tm = summary["test_metrics"]
print("=" * 60)
print("===== COPY FROM HERE — ye block Muse ko bhej dena =====")
print(f"RUN OK — {RUN_NAME}")
print(f"Weights file : {dated.name}")
print(f"Size (MB)    : {dated.stat().st_size / 1e6:.2f}")
print(f"SHA-256      : {sha}")
print(f"Test mAP@0.5 : {tm.get('mAP50')} | mAP@0.5:0.95 : {tm.get('mAP50_95')} | P : {tm.get('precision')} | R : {tm.get('recall')} | F1 : {tm.get('f1')}")
print(f"vs v2 baseline mAP@0.5 : {V2_BASELINE_MAP50} | Gate: {'PASS' if (tm.get('mAP50') or 0) > V2_BASELINE_MAP50 else 'FAIL — v2 weights stay the fallback'}")
for cls, vals in tm.get("per_class", {}).items():
    print(f"  {cls}: AP50 = {vals.get('AP50')}")
print("===== COPY TO HERE =====")
print("=" * 60)
print(f"Weights + results saved in: {dest}  (Kaggle Output tab se download kar lo)")